In [5]:
import os
import glob
import logging
import time
import pandas as pd
from io import BytesIO
from PIL import Image
import pytesseract
# Indica la ruta al ejecutable de Tesseract en Windows:
pytesseract.pytesseract.tesseract_cmd = r"D:\Program Files\tesseract.exe"
from concurrent.futures import ThreadPoolExecutor
import certifi

# ——————————————————————————————————————————————
# 1. GLOBAL CONFIGURATION
# ——————————————————————————————————————————————
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)

# ——————————————————————————————————————————————
# 2. LOCAL OCR (TESSERACT)
# ——————————————————————————————————————————————
# Asegúrate de haber instalado Tesseract OCR en tu máquina.
# Puedes descargarlo desde: https://github.com/tesseract-ocr/tesseract

def ocr_single_image_local(raw_bytes, language="eng"):
    """
    Ejecuta OCR sobre los bytes de imagen usando Tesseract.
    """
    try:
        img = Image.open(BytesIO(raw_bytes))
        text = pytesseract.image_to_string(img, lang=language)
        return text.strip() if text else "<no text>"
    except Exception as e:
        logging.error(f"OCR failed: {e}")
        return f"<OCR ERROR: {e}>"

# ——————————————————————————————————————————————
# 3. LOCAL PARQUET DIRECTORY
# ——————————————————————————————————————————————
PARQUET_DIR = (
    r"C:\Users\ninic\OneDrive - Lambton College\Term 3"
    r"\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)"
    r"\Project Folder\Week 1-2-Kick Off\parquet files"
)

# ——————————————————————————————————————————————
# 4. HELPER FUNCTIONS
# ——————————————————————————————————————————————
def list_parquet_files(dir_path):
    files = glob.glob(os.path.join(dir_path, "*.parquet"))
    logging.info(f"Found {len(files)} Parquet files in '{dir_path}'.")
    return files

def extract_image_info(row):
    img_struct = row["image"]
    raw_bytes  = img_struct["bytes"]
    name_field = img_struct["path"]
    name = (
        name_field.decode("utf-8", errors="ignore")
        if isinstance(name_field, (bytes, bytearray))
        else name_field
    )
    img = Image.open(BytesIO(raw_bytes))
    w, h = img.size
    return raw_bytes, name, img.format.upper(), w, h

# ——————————————————————————————————————————————
# 5. PROCESAMIENTO PARALLELO
# ——————————————————————————————————————————————
def process_row(row):
    raw, name, fmt, w, h = extract_image_info(row)
    text_ = ocr_single_image_local(raw)
    return {
        "Label":    row.get("label"),
        "Path":     name,
        "Format":   fmt,
        "Size":     f"{w} x {h} px",
        "OCR_Text": text_
    }

def process_and_collect_parallel(parquet_dir, max_workers=16):
    files  = list_parquet_files(parquet_dir)
    records = []
    for fp in files:
        logging.info(f"Processing Parquet: {fp}")
        try:
            df = pd.read_parquet(fp, engine="pyarrow")
        except Exception as e:
            logging.error(f"Failed to read '{fp}': {e}")
            continue
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            for rec in executor.map(process_row, [row for _, row in df.iterrows()]):
                records.append(rec)
    return records

# ——————————————————————————————————————————————
# 6. MAIN EXECUTION
# ——————————————————————————————————————————————
if __name__ == "__main__":
    records = process_and_collect_parallel(PARQUET_DIR, max_workers=16)
    df = pd.DataFrame(records)
    # Guardar el DataFrame en un CSV local
    df.to_csv(r"C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\Nicolascode\ocr_results.csv", index=False)
    logging.info("Results saved to CSV at 'C:\\path\\to\\save\\ocr_results.csv'.")
    display(df)


2025-05-26 11:33:04,855 [INFO] Found 10 Parquet files in 'C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\parquet files'.
2025-05-26 11:33:04,855 [INFO] Processing Parquet: C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\parquet files\0000.parquet
2025-05-26 11:37:59,939 [INFO] Processing Parquet: C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\parquet files\0001.parquet
2025-05-26 11:42:49,141 [INFO] Processing Parquet: C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3 BDM 3035 - Big Data Capstone Project 01 (DSMM Group 1)\Project Folder\Week 1-2-Kick Off\parquet files\0002.parquet
2025-05-26 11:48:07,832 [INFO] Processing Parquet: C:\Users\ninic\OneDrive - Lambton College\Term 3\2025S-T3

,Label,Path,Format,Size,OCR_Text
0,0,00163fe7688e71ce06f495a6811fef71_1.jpg,JPEG,1224 x 1584 px,WIKIPEDIA\n\n: Donate Create account Log in\n‘...
1,0,00163fe7688e71ce06f495a6811fef71_2.jpg,JPEG,1224 x 1584 px,Animation {ecit]\n\nYear Title Role Notes Sour...
2,0,00163fe7688e71ce06f495a6811fef71_3.jpg,JPEG,1224 x 1584 px,Pretty Pretty Please | Don't | Sidney the . .\...
3,0,00163fe7688e71ce06f495a6811fef71_4.jpg,JPEG,1224 x 1584 px,"Brooke Page, Darling\n\n2015 | Ever After High..."
4,0,00163fe7688e71ce06f495a6811fef71_5.jpg,JPEG,1224 x 1584 px,Live-action [edit]\nYear Title Role Notes Sour...
...,...,...,...,...,...
9095,1,d4359c96a990546ce443842f13e6135c_10.jpg,JPEG,1224 x 1584 px,"* Azerbaijan, Belarus, Kazakhstan, Kyrgyzstan,..."
9096,1,d4359c96a990546ce443842f13e6135c_11.jpg,JPEG,1224 x 1584 px,Occupied by Russia\nCrimea before an illegal\n...
9097,1,d4359c96a990546ce443842f13e6135c_12.jpg,JPEG,1224 x 1584 px,wars [ecit)\n\nSee also: Post-Soviet conflicts...
9098,1,d4359c96a990546ce443842f13e6135c_13.jpg,JPEG,1224 x 1584 px,Rasputin fostered an environmentalist sentimen...
